<a href="https://colab.research.google.com/github/charansai12322/currencyconverter/blob/main/currencyconerter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install -U google-genai requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 9.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.61.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [8]:
from google import genai
from google.genai import types
from google.colab import userdata

import requests
import re
from datetime import datetime, timezone

# Read the key securely from Colab Secrets.
GEMINI_API_KEY = userdata.get("sec1")

if not GEMINI_API_KEY:
    raise ValueError(
        "Could not find GEMINI_API_KEY. Add it in Colab's Secrets panel "
        "and enable notebook access."
    )

client = genai.Client(api_key=GEMINI_API_KEY)

# Change this if you want to use another Gemini model available to your API key.
MODEL_NAME = "gemini-3.5-flash"

print("Gemini client is ready.")

Gemini client is ready.


In [9]:
def convert_currency(amount: float, from_currency: str, to_currency: str) -> dict:
    """
    Converts money between two ISO 4217 currency codes using live exchange rates.

    Args:
        amount: Amount of money to convert. Example: 100
        from_currency: Three-letter source currency code. Example: USD
        to_currency: Three-letter target currency code. Example: INR

    Returns:
        A dictionary with the conversion result, exchange rate, and rate timestamp.
    """

    from_currency = from_currency.strip().upper()
    to_currency = to_currency.strip().upper()

    # Basic safety/format checks before making the API request.
    currency_pattern = r"^[A-Z]{3}$"

    if not re.fullmatch(currency_pattern, from_currency):
        return {
            "error": f"'{from_currency}' is not a valid three-letter currency code."
        }

    if not re.fullmatch(currency_pattern, to_currency):
        return {
            "error": f"'{to_currency}' is not a valid three-letter currency code."
        }

    try:
        amount = float(amount)
    except (TypeError, ValueError):
        return {"error": "Amount must be a number."}

    if amount < 0:
        return {"error": "Amount cannot be negative."}

    # Free live-rate endpoint. Rates are expressed relative to from_currency.
    url = f"https://open.er-api.com/v6/latest/{from_currency}"

    try:
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        data = response.json()
    except requests.RequestException as error:
        return {
            "error": "Could not reach the exchange-rate service.",
            "details": str(error),
        }

    if data.get("result") != "success":
        return {
            "error": f"Could not retrieve rates for {from_currency}.",
            "details": data.get("error-type", "Unknown API error"),
        }

    rates = data.get("rates", {})

    if to_currency not in rates:
        return {
            "error": (
                f"Currency code '{to_currency}' was not found. "
                "Use a standard ISO currency code such as USD, INR, EUR, GBP, or JPY."
            )
        }

    exchange_rate = float(rates[to_currency])
    converted_amount = amount * exchange_rate

    return {
        "success": True,
        "original_amount": amount,
        "from_currency": from_currency,
        "to_currency": to_currency,
        "exchange_rate": exchange_rate,
        "converted_amount": round(converted_amount, 2),
        "rate_last_updated_utc": data.get("time_last_update_utc"),
        "rate_next_update_utc": data.get("time_next_update_utc"),
        "source": "open.er-api.com",
    }

In [10]:
# This describes the Python tool to Gemini.
currency_tool_declaration = {
    "name": "convert_currency",
    "description": (
        "Convert an amount from one currency to another using current live exchange rates. "
        "Use this whenever the user asks for a currency conversion, exchange rate, "
        "or the value of money in another currency."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "amount": {
                "type": "number",
                "description": "The amount of money to convert. Example: 100",
            },
            "from_currency": {
                "type": "string",
                "description": "Three-letter source ISO currency code. Example: USD",
            },
            "to_currency": {
                "type": "string",
                "description": "Three-letter target ISO currency code. Example: INR",
            },
        },
        "required": ["amount", "from_currency", "to_currency"],
    },
}

tools = [
    types.Tool(function_declarations=[currency_tool_declaration])
]

config = types.GenerateContentConfig(
    tools=tools,
    system_instruction=(
        "You are a helpful Currency Converter Assistant. "
        "For every question requiring a current conversion or exchange rate, "
        "call convert_currency before answering. "
        "Use the exact result returned by the tool. "
        "Clearly state that exchange rates can change."
    ),
)

available_functions = {
    "convert_currency": convert_currency
}

conversation = []


def ask_currency_assistant(user_message: str, max_tool_rounds: int = 5) -> str:
    """
    Sends a message to Gemini and automatically executes requested tools.
    Conversation history is kept in the global `conversation` list.
    """

    global conversation

    # Add the user's new message to the ongoing conversation.
    conversation.append(
        types.Content(
            role="user",
            parts=[types.Part(text=user_message)]
        )
    )

    for _ in range(max_tool_rounds):
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=conversation,
            config=config,
        )

        function_calls = response.function_calls or []

        # If Gemini did not request a tool, it has produced the final answer.
        if not function_calls:
            final_text = response.text or "Sorry, I could not generate a response."
            conversation.append(response.candidates[0].content)
            return final_text

        # Preserve Gemini's tool-call message in the conversation.
        conversation.append(response.candidates[0].content)

        function_response_parts = []

        # Gemini can request more than one tool call in a turn.
        for tool_call in function_calls:
            tool_name = tool_call.name
            tool_args = dict(tool_call.args)

            print(f"🔧 Tool call: {tool_name}({tool_args})")

            if tool_name in available_functions:
                try:
                    result = available_functions[tool_name](**tool_args)
                except Exception as error:
                    result = {
                        "error": "The currency tool had an unexpected error.",
                        "details": str(error),
                    }
            else:
                result = {
                    "error": f"Unknown tool requested: {tool_name}"
                }

            print(f"📈 Tool result: {result}\n")

            # Return the result to Gemini using the matching call ID.

            function_response_parts.append(
                types.Part.from_function_response(
                    name=tool_name,
                    response={"result": result},
                )
            )



        conversation.append(
            types.Content(
                role="user",
                parts=function_response_parts,
            )
        )

    return "The assistant reached its tool-call limit. Please try asking again."


def reset_conversation():
    """Clears the assistant's chat memory."""
    global conversation
    conversation = []
    print("Conversation reset.")

In [11]:
print(ask_currency_assistant("Convert 100 US dollars to Indian rupees."))

🔧 Tool call: convert_currency({'to_currency': 'INR', 'amount': 100, 'from_currency': 'USD'})
📈 Tool result: {'success': True, 'original_amount': 100.0, 'from_currency': 'USD', 'to_currency': 'INR', 'exchange_rate': 96.867192, 'converted_amount': 9686.72, 'rate_last_updated_utc': 'Sat, 10 Oct 2026 00:02:31 +0000', 'rate_next_update_utc': 'Sun, 11 Oct 2026 00:10:21 +0000', 'source': 'open.er-api.com'}

100 US Dollars (USD) is currently equal to **9,686.72 Indian Rupees (INR)** at an exchange rate of 1 USD = 96.867192 INR.

*Please note that exchange rates fluctuate frequently and can change over time.*
